## PDF to Images

In [5]:
import os
import fitz  # PyMuPDF
from pathlib import Path

def extract_pdf_pages_as_images(data_folder="data", output_folder="images"):
    """
    Extract all pages from PDF files in the data folder as images and save them
    to the output folder, organized by PDF filename.
    
    Args:
        data_folder (str): Path to folder containing PDF files
        output_folder (str): Path to save images
    """
    # Ensure the data folder exists
    if not os.path.exists(data_folder):
        print(f"Data folder '{data_folder}' does not exist.")
        return
        
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # Find all PDF files in the data folder
    pdf_files = [f for f in os.listdir(data_folder) if f.lower().endswith('.pdf')]
    
    if not pdf_files:
        print(f"No PDF files found in '{data_folder}'.")
        return
        
    print(f"Found {len(pdf_files)} PDF files. Starting extraction...")
    
    # Process each PDF file
    for pdf_file in pdf_files:
        pdf_path = os.path.join(data_folder, pdf_file)
        pdf_name = os.path.splitext(pdf_file)[0]
        
        # Create a folder for this PDF's images
        pdf_output_folder = os.path.join(output_folder, pdf_name)
        if not os.path.exists(pdf_output_folder):
            os.makedirs(pdf_output_folder)
            
        print(f"Processing: {pdf_file}")
        
        try:
            # Open the PDF file
            doc = fitz.open(pdf_path)
            
            # Process each page
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                
                # Render page to an image (300 DPI)
                pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
                
                # Save the image
                output_file = os.path.join(pdf_output_folder, f"{pdf_name}_{page_num+1}.png")
                pix.save(output_file)
                
            print(f"  Extracted {len(doc)} pages from {pdf_file}")
            doc.close()
            
        except Exception as e:
            print(f"  Error processing {pdf_file}: {str(e)}")
            
    print(f"Extraction complete. Images saved to '{output_folder}' directory.")

if __name__ == "__main__":
    extract_pdf_pages_as_images()

Found 13 PDF files. Starting extraction...
Processing: IEXT.pdf
  Extracted 3 pages from IEXT.pdf
Processing: GRAPHICS AND INDUSTRIAL CIRCUITS.pdf
  Extracted 4 pages from GRAPHICS AND INDUSTRIAL CIRCUITS.pdf
Processing: Alpine Bearing Company, Inc..pdf
  Extracted 1 pages from Alpine Bearing Company, Inc..pdf
Processing: SUBSEA PROTECTION SYSTEMS.pdf
  Extracted 1 pages from SUBSEA PROTECTION SYSTEMS.pdf
Processing: LAKELAND ELECTRICAL MOTOR SERVICES, INC..pdf
  Extracted 1 pages from LAKELAND ELECTRICAL MOTOR SERVICES, INC..pdf
Processing: Cast Metals Technology.pdf
  Extracted 3 pages from Cast Metals Technology.pdf
Processing: AESSEAL INC.pdf
  Extracted 1 pages from AESSEAL INC.pdf
Processing: Stanley M. Proctor Company.pdf
  Extracted 1 pages from Stanley M. Proctor Company.pdf
Processing: PROMED MOLDED PRODUCTS INC.pdf
  Extracted 7 pages from PROMED MOLDED PRODUCTS INC.pdf
Processing: EMJ Credit East.pdf
  Extracted 1 pages from EMJ Credit East.pdf
Processing: STANDARD ELECTRIC

## Text to Json

In [3]:
import os
import json
import re
from datetime import datetime
from pathlib import Path

def parse_text_file(file_path):
    """
    Parse a text file with financial data into a structured dictionary.
    """
    with open(file_path, 'r') as file:
        content = file.read()
    
    # Split the content by lines
    lines = content.strip().split('\n')
    
    # Initialize dictionary for storing the parsed data
    data = {}
    
    # Parse key-value pairs at the beginning
    i = 0
    while i < len(lines):
        if lines[i] == "Balance Assertion":
            i += 1
            break
        
        if i + 1 < len(lines):
            key = lines[i]
            value = lines[i + 1]
            
            # Convert dates to standard format
            if key in ["Opening Date", "Closing Date"]:
                try:
                    # Validate date format
                    datetime.strptime(value, "%Y-%m-%d")
                except ValueError:
                    pass
            
            # Convert numeric values to float
            elif key in ["Opening Balance", "Closing Balance"]:
                try:
                    value = float(value)
                except ValueError:
                    pass
            
            data[key] = value
            i += 2
        else:
            i += 1
    
    # Process the table after "Balance Assertion"
    if i < len(lines) and lines[i].startswith("Date"):
        # Get headers
        headers = lines[i].split('\t')
        
        # Process transactions
        transactions = []
        for j in range(i + 1, len(lines)):
            if lines[j].strip():  # Skip empty lines
                values = lines[j].split('\t')
                if len(values) == len(headers):
                    transaction = {}
                    for k, header in enumerate(headers):
                        value = values[k]
                        
                        # Convert date
                        if header == "Date":
                            try:
                                # Validate date format
                                datetime.strptime(value, "%Y-%m-%d")
                            except ValueError:
                                pass
                        
                        # Convert numeric values to float
                        elif header in ["Amount", "Balance"]:
                            try:
                                value = float(value)
                            except ValueError:
                                pass
                        
                        transaction[header] = value
                    
                    transactions.append(transaction)
        
        data["Transactions"] = transactions
    
    return data

def convert_txt_to_json(input_folder, output_folder):
    """
    Convert all text files in the input folder to JSON files in the output folder.
    """
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Find all text files in the input folder
    text_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.txt')]
    
    if not text_files:
        print(f"No text files found in '{input_folder}'.")
        return
    
    print(f"Found {len(text_files)} text files. Starting conversion...")
    
    # Process each text file
    for txt_file in text_files:
        try:
            # Parse the text file
            input_path = os.path.join(input_folder, txt_file)
            data = parse_text_file(input_path)
            
            # Save as JSON
            output_file = os.path.splitext(txt_file)[0] + '.json'
            output_path = os.path.join(output_folder, output_file)
            
            with open(output_path, 'w') as json_file:
                json.dump(data, json_file, indent=2)
            
            print(f"Converted: {txt_file} -> {output_file}")
            
        except Exception as e:
            print(f"Error processing {txt_file}: {str(e)}")
    
    print(f"Conversion complete. JSON files saved to '{output_folder}' directory.")


    
convert_txt_to_json("data", "json_output")

Found 13 text files. Starting conversion...
Converted: SUBSEA PROTECTION SYSTEMS.txt -> SUBSEA PROTECTION SYSTEMS.json
Converted: Inspection Consultants Ltd.txt -> Inspection Consultants Ltd.json
Converted: LAKELAND ELECTRICAL MOTOR SERVICES, INC.txt -> LAKELAND ELECTRICAL MOTOR SERVICES, INC.json
Converted: Cast Metals Technology.txt -> Cast Metals Technology.json
Converted: IEXT.txt -> IEXT.json
Converted: KIRSH FOUNDRY.txt -> KIRSH FOUNDRY.json
Converted: EMJ Credit East.txt -> EMJ Credit East.json
Converted: Stanley M. Proctor Company.txt -> Stanley M. Proctor Company.json
Converted: STANDARD ELECTRIC SUPPLY CO - PDQ.txt -> STANDARD ELECTRIC SUPPLY CO - PDQ.json
Converted: AESSEAL INC.txt -> AESSEAL INC.json
Converted: Alpine Bearing Company, Inc.txt -> Alpine Bearing Company, Inc.json
Converted: GRAPHICS AND INDUSTRIAL CIRCUITS.txt -> GRAPHICS AND INDUSTRIAL CIRCUITS.json
Converted: PROMED MOLDED PRODUCTS INC.txt -> PROMED MOLDED PRODUCTS INC.json
Conversion complete. JSON files s

## Creating HuggingFace dataset

In [22]:
from datasets import Dataset, DatasetDict
import os
import glob
import random
from PIL import Image
import json

# Set seed for reproducibility
random.seed(42)

# Define paths and parameters
base_dir = "transformed_data"
train_folders_count = 10
test_folders_count = 3

# Get list of all PDF folders
pdf_folders = [os.path.join(base_dir, d) for d in os.listdir(base_dir) 
               if os.path.isdir(os.path.join(base_dir, d))]

# Shuffle folders to ensure random split
# random.shuffle(pdf_folders)

# Split folders into train and test
train_folders = pdf_folders[:train_folders_count]
test_folders = pdf_folders[train_folders_count:train_folders_count+test_folders_count]

def create_split_dataset(folders):
    """Create dataset split from given folders"""
    examples = []
    image_extensions = ('*.png', '*.jpg', '*.jpeg')
    
    for folder in folders:
        # Get all image paths in folder
        image_paths = []
        for ext in image_extensions:
            image_paths.extend(glob.glob(os.path.join(folder, ext)))
        
        # Create image-json pairs
        for img_path in image_paths:
            json_path = os.path.splitext(img_path)[0] + '.json'
            
            if os.path.exists(json_path):
                # Load image and JSON data
                with open(json_path, 'r') as f:
                    json_data = json.load(f)
                
                examples.append({
                    'image': Image.open(img_path).convert("RGB"),
                    'json': json_data
                })
    
    return Dataset.from_list(examples)

# Create train and test datasets
train_dataset = create_split_dataset(train_folders)
test_dataset = create_split_dataset(test_folders)

# Create DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})


In [ ]:
from huggingface_hub import login

# Login into Hugging Face Hub
login(os.getenv("HUGGINGFACE_TOKEN"))
dataset.push_to_hub("ikram98ai/invoice_img2json")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/695 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ikram98ai/invoice_img2json/commit/551edcb7728a90d0c7f91596031d5eb46d69f8c5', commit_message='Upload dataset', commit_description='', oid='551edcb7728a90d0c7f91596031d5eb46d69f8c5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ikram98ai/invoice_img2json', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ikram98ai/invoice_img2json'), pr_revision=None, pr_num=None)

In [ ]:
# (Optional) Save dataset locally
dataset.save_to_disk("image_json_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/27 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]